In [15]:
from google.colab import files
import pandas as pd
import io
import numpy as np


uploaded = files.upload()

# Get the filename dynamically to avoid errors
filename = list(uploaded.keys())[0]

# Load with professional column names
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']
df = pd.read_csv(io.BytesIO(uploaded[filename]), names=columns)

df.head()

Saving diabetes.data.csv to diabetes.data (3).csv


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [16]:
#  Replace impossible 0s with Median )
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_fix] = df[cols_to_fix].replace(0, np.nan)
df.fillna(df.median(), inplace=True)


df['Glucose_Age_Ratio'] = df['Glucose'] / (df['Age'] + 1)
df['Health_Score'] = (df['BMI'] * df['Glucose']) / 100

print("✨ Features Cleaned & Engineered. Data is now high-quality.")

✨ Features Cleaned & Engineered. Data is now high-quality.


In [17]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import joblib

#Prepare Features and Target
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 3. Hyperparameter Tuning for 80%+ Accuracy
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'class_weight': ['balanced', None]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc')
grid.fit(X_train_scaled, y_train)

# 4. Export the Brain
best_model = grid.best_estimator_
joblib.dump(best_model, 'disease_model.joblib')
joblib.dump(scaler, 'disease_scaler.joblib')

print(f" Model Tuned! Best CV Score: {grid.best_score_:.4f}")
print("DOWNLOAD NOW: Look in the left sidebar for 'disease_model.joblib' and 'disease_scaler.joblib'")

 Model Tuned! Best CV Score: 0.8306
DOWNLOAD NOW: Look in the left sidebar for 'disease_model.joblib' and 'disease_scaler.joblib'
